# LeNet-5 NVIDIA GPU Benchmark (Colab)
Run this notebook with **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU).

This notebook produces **new** measurements for PyTorch CUDA inference and an educational CUDA convolution kernel. It does not automatically reproduce the historical AutoTVM training result. See `docs/SETUP.md` for the legacy TVM pipeline.


In [ ]:
import os, subprocess, json
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU runtime in Colab first'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
!git clone https://github.com/fardeenshroff/lenet5-gpu-autotvm-optimization.git
%cd lenet5-gpu-autotvm-optimization
!pip -q install -e .
!python -m pytest -q


## Train a new LeNet-5 checkpoint on MNIST
This is a new training run; accuracy and training times depend on the current runtime and optimizer settings.


In [ ]:
!python scripts/train.py --device cuda --epochs 1 --batch-size 256 --output checkpoints/colab_cuda.pt --metrics results/colab_cuda_training.json


## Benchmark inference with same checkpoint and batch size
CPU and GPU measured in the same Colab runtime, but distinct devices. A synchronized PyTorch inference measurement includes Python dispatch and is not a raw CUDA-kernel measurement.


In [ ]:
!python scripts/benchmark_pytorch.py --device cpu --checkpoint checkpoints/colab_cuda.pt --batch-size 128 --warmup 30 --iterations 200 --output results/colab_cpu_inference.json
!python scripts/benchmark_pytorch.py --device cuda --checkpoint checkpoints/colab_cuda.pt --batch-size 128 --warmup 30 --iterations 200 --output results/colab_cuda_inference.json


## Compile and time a standalone CUDA/C++ kernel
This kernel validates output against a CPU reference. It is intentionally a simple learning example, not a new claim about the original AutoTVM experiment.


In [ ]:
!nvcc -O3 -std=c++14 cuda/conv2d_naive.cu -o /tmp/conv2d_naive
!/tmp/conv2d_naive


In [ ]:
from pathlib import Path
cpu = json.loads(Path('results/colab_cpu_inference.json').read_text())
gpu = json.loads(Path('results/colab_cuda_inference.json').read_text())
report = {'comparison': 'new synchronized PyTorch inference', 'batch_size': cpu['batch_size'], 'gpu': torch.cuda.get_device_name(0), 'cpu_median_ms': cpu['median_ms'], 'gpu_median_ms': gpu['median_ms'], 'cpu_to_cuda_inference_speedup': cpu['median_ms']/gpu['median_ms']}
Path('results/colab_cpu_vs_cuda.json').write_text(json.dumps(report, indent=2)+'\n')
print(json.dumps(report, indent=2))
print('Share these new JSON results for a verified GitHub update.')
